# Installation

In [1]:
%pip install -q llmcompressor datasets transformers accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Configuration

In [2]:
from dataclasses import dataclass

@dataclass
class Config:
    # Model choices: you can add more HF IDs here
    model_id: str = "unsloth/llama-3.2-1b-Instruct"
    # model_id: str = "gpt2"
    # model_id: str = "Qwen/Qwen2.5-7B-Instruct"

    # Quantization method: "gptq", "awq", or "data_free"
    quant_method: str = "gptq"

    # Calibration dataset choice
    # "open_platypus", "ultrachat", "c4_small", "wikitext", or None for data-free
    cal_dataset: str = "open_platypus"

    num_calibration_samples: int = 128
    max_seq_length: int = 512

    # Quantization scheme
    # For GPTQ/AWQ via llmcompressor
    scheme: str = "W4A16"

    # Output directory
    save_dir: str = "quantized_model"

cfg = Config()
print(cfg)

Config(model_id='unsloth/llama-3.2-1b-Instruct', quant_method='gptq', cal_dataset='open_platypus', num_calibration_samples=128, max_seq_length=512, scheme='W4A16', save_dir='quantized_model')


# Load Model & Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    torch_dtype="auto",
    device_map="auto",
    token=os.environ.get("HF_TOKEN", None),
)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)

e:\PhD\projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'os' is not defined

# Calibration Dataset Loader

In [ ]:
from datasets import load_dataset

def load_calibration_dataset(cfg, tokenizer):
    """
    Returns a tokenized calibration dataset or None (for data-free PTQ).
    """
    if cfg.cal_dataset is None:
        print("No calibration dataset selected (data-free quantization).")
        return None

    if cfg.cal_dataset == "open_platypus":
        ds = load_dataset("open_platypus", split="train")
        ds = ds.shuffle(seed=42).select(range(cfg.num_calibration_samples))
        # Assume 'question' + 'answer' style; adapt as needed
        def to_text(example):
            return {"text": example.get("question", "") + "\n" + example.get("answer", "")}
        ds = ds.map(to_text)

    elif cfg.cal_dataset == "ultrachat":
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
        ds = ds.shuffle(seed=42).select(range(cfg.num_calibration_samples))
        def preprocess(example):
            return {
                "text": tokenizer.apply_chat_template(
                    example["messages"],
                    tokenize=False,
                )
            }
        ds = ds.map(preprocess)

    elif cfg.cal_dataset == "c4_small":
        ds = load_dataset("brando/small-c4-dataset", split="train")
        ds = ds.shuffle(seed=42).select(range(cfg.num_calibration_samples))
        ds = ds.rename_column("text", "text")

    elif cfg.cal_dataset == "wikitext":
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")
        ds = ds.shuffle(seed=42).select(range(cfg.num_calibration_samples))
        ds = ds.rename_column("text", "text")

    else:
        raise ValueError(f"Unknown calibration dataset: {cfg.cal_dataset}")

    def tokenize(sample):
        return tokenizer(
            sample["text"],
            padding=False,
            max_length=cfg.max_seq_length,
            truncation=True,
            add_special_tokens=False,
        )

    ds = ds.map(tokenize, remove_columns=[c for c in ds.column_names if c != "input_ids"])
    return ds

calib_ds = load_calibration_dataset(cfg, tokenizer)

# Quantization (GPTQ / AWQ)

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.gptq import GPTQModifier
try:
    from llmcompressor.modifiers.awq import AWQModifier
except ImportError:
    AWQModifier = None
    print("AWQModifier not available in this llmcompressor version.")

def build_recipe(cfg):
    if cfg.quant_method == "gptq":
        print("Using GPTQ quantization.")
        return GPTQModifier(
            targets=["Linear"],
            scheme=cfg.scheme,
            ignore=["lm_head"],
        )
    elif cfg.quant_method == "awq":
        if AWQModifier is None:
            raise RuntimeError("AWQModifier not available. Please update llmcompressor.")
        print("Using AWQ quantization.")
        return AWQModifier(
            targets=["Linear"],
            scheme=cfg.scheme,
            ignore=["lm_head"],
        )
    else:
        raise ValueError(f"Unsupported quant_method for oneshot: {cfg.quant_method}")

if cfg.quant_method in ["gptq", "awq"]:
    recipe = build_recipe(cfg)

    oneshot(
        model=model,
        dataset=calib_ds,
        recipe=recipe,
        max_seq_length=cfg.max_seq_length,
        num_calibration_samples=cfg.num_calibration_samples,
    )

    quant_save_dir = f"{cfg.model_id.replace('/', '_')}-{cfg.scheme}-{cfg.quant_method}"
    model.save_pretrained(quant_save_dir, save_compressed=True)
    tokenizer.save_pretrained(quant_save_dir)
    print(f"Done! Quantized model saved to {quant_save_dir}")

# Data-Free PTQ

In [ ]:
from llmcompressor import model_free_ptq

if cfg.quant_method == "data_free":
    quant_save_dir = f"{cfg.model_id.replace('/', '_')}-{cfg.scheme}-data_free"
    model_free_ptq(
        model_stub=cfg.model_id,
        save_directory=quant_save_dir,
        scheme=cfg.scheme,
    )
    tokenizer.save_pretrained(quant_save_dir)
    print(f"Done! Data-free quantized model saved to {quant_save_dir}")

# Sources



*   https://github.com/vllm-project/llm-compressor
*   https://huggingface.co/unsloth
*   https://docs.vllm.ai/projects/llm-compressor/en/latest/guides/entrypoints/model-free-ptq/#when-to-use
*   https://huggingface.co/datasets/allenai/c4
*   https://huggingface.co/datasets/brando/small-c4-dataset
*   https://huggingface.co/datasets/Salesforce/wikitext

